# Task 2: Fine-Tuning LLM for Tax Law using QLoRA
## Parameter-Efficient Fine-Tuning (PEFT)

**Objective:** Fine-tune a base model (`google/gemma-4-E2B-it`) on a domain-specific dataset (`DomainLLM/german-law-qa`) using QLoRA to create a specialized tax and law skill.

**Technical Approach (Based on Course Requirements):**
* **Quantization:** Load base model in 4-bit (Adapter Quantization / compression).
* **Freezing:** Freeze the base model weights.
* **PEFT / LoRA:** Inject a 16-bit LoRA adapter and train only these parameters.
* **Environment:** Google Colab (T4 GPU).


### Step 1: Install Required Libraries
First, we install the Hugging Face ecosystem libraries necessary for QLoRA.

In [1]:
# Install required libraries for quantization, fine-tuning, and data handling
# Install Unsloth, Xformers (for memory efficient attention) and other dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets pandas tqdm huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_zsxngjw/unsloth_815513e373ad416ebad323913e59d0d3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_zsxngjw/unsloth_815513e373ad416ebad323913e59d0d3
  Resolved https://github.com/unslothai/unsloth.git to commit f801e59c29db7b4028297ef987af6bfdaa464500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.4 MB/s eta 0:00:00


### Step 2: Authentication

Authenticate with Hugging Face to access the base model

In [2]:
from huggingface_hub import notebook_login

# Log in to Hugging Face (Requires a token with 'Write' permissions)
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Step 3: Load the Dataset
We load the DomainLLM/german-law-qa dataset. We also define a formatting function to convert the question and answer into the chat template that Gemma expects.

In [3]:
from datasets import load_dataset

# Load the target dataset
dataset_id = "DomainLLM/german-law-qa"
dataset = load_dataset(dataset_id, split="train")

def format_prompt(example):
    """Formats the prompt into Gemma's instruction format."""
    question = example['question']
    answer = example['answer']

    # Gemma chat template formatting
    text = f"<bos><start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn><eos>"
    return {"text": text}

# Apply formatting to the dataset
formatted_dataset = dataset.map(format_prompt)
print("Dataset formatting complete. Example:")
print(formatted_dataset[0]['text'])

README.md:   0%|          | 0.00/705 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.88M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/911k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14160 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map:   0%|          | 0/14160 [00:00<?, ? examples/s]

Dataset formatting complete. Example:
<bos><start_of_turn>user
Welches Ziel muss der Täter nach § 107c StGB verfolgen, damit eine Strafbarkeit eintritt?<end_of_turn>
<start_of_turn>model
Der Täter muss nach § 107c StGB mit der Absicht handeln, sich oder einem anderen Kenntnis darüber zu verschaffen, wie jemand gewählt hat.<end_of_turn><eos>


### Step 4: Load Base Model in 4-bit (Quantization)
Following the QLoRA methodology: Base model -> quantization -> 4-bit -> freeze.

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # Maximum sequence length. Unsloth auto-supports RoPE scaling.
dtype = None # None for auto detection. Float16 for Tesla T4
load_in_4bit = True # Use 4bit quantization to reduce memory usage.

model_id = "google/gemma-4-E2B-it" # Your specific base model

print("Loading model using Unsloth optimized kernels...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model using Unsloth optimized kernels...
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

'The read operation timed out' thrown while requesting HEAD https://huggingface.co/google/gemma-4-E2B-it/resolve/main/preprocessor_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model loaded successfully!


### Step 5: Initialize LoRA Adapter (16-bit)
We attach the 16-bit LoRA adapter. We will only train these specific projection layers.

In [5]:
print("Injecting LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 8,
    lora_dropout = 0, # Unsloth specifically optimizes for dropout = 0
    bias = "none",    # Unsloth specifically optimizes for bias = "none"
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()

Injecting LoRA adapters...
trainable params: 4,644,864 || all params: 5,127,822,880 || trainable%: 0.0906


### Step 6: Train the Model
We use SFTTrainer (Supervised Fine-Tuning Trainer) to train our new "AT tax skill".

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

print("Configuring Unsloth Trainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 150, # Set to a low number for quick fine-tuning in Colab
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit", # Use 8bit optimizer to save memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training process...")
trainer_stats = trainer.train()
print("Training completed!")

Configuring Unsloth Trainer...


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/14160 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


Starting training process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,160 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 4,644,864 of 5,127,822,880 (0.09% trained)


Step,Training Loss
10,3.511942
20,2.640980
30,1.978838
40,1.642767
50,1.516996
60,1.388969
70,1.325918
80,1.325372
90,1.311111
100,1.349121


Training completed!


### Step 7: Save the Fine-Tuned Adapter
After training, we save the LoRA weights locally. For inference, these weights will be merged with the base model on the fly.

In [7]:
import os

save_path = "./Team1/results/lora_adapter"
os.makedirs(save_path, exist_ok=True)

print("Saving LoRA adapters...")
model.save_pretrained(save_path) # Local saving
tokenizer.save_pretrained(save_path)
print(f"Adapters saved to {save_path}")

Saving LoRA adapters...
Adapters saved to ./Team1/results/lora_adapter


### Step 8: Inference on Test Set and CSV Generation

In [8]:
import pandas as pd
import os
from tqdm import tqdm

INPUT_FILE = "dataset_clean.csv"
OUTPUT_CSV = "./Team1/results/model2_finetuned.csv"

# Ensure the directory for saving the file exists
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

if not os.path.exists(INPUT_FILE):
    print(f"ERROR: Please upload {INPUT_FILE} to Colab first!")
else:
    test_df = pd.read_csv(INPUT_FILE)
    results = []

    print(f"Starting optimized Unsloth inference on {len(test_df)} questions...")

    # Enable native 2x faster inference
    FastLanguageModel.for_inference(model)

    # Iterate over all rows
    for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
        q_id = row['id']
        prompt_text = str(row['prompt']) # Protect against empty strings (NaN)

        # Format prompt
        chat_template = f"<bos><start_of_turn>user\n{prompt_text}<end_of_turn>\n<start_of_turn>model\n"

        # Tokenize (explicitly using text=)
        inputs = tokenizer(text=[chat_template], return_tensors="pt").to("cuda")

        # Generate
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            use_cache=True, # Unsloth optimization
            temperature=0.1,
        )

        # Extract answer
        input_length = inputs['input_ids'].shape[1]
        generated_tokens = outputs[0][input_length:]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Save to list
        results.append({
            "id": q_id,
            "answer": answer
        })

        # Save intermediate progress every 5 questions
        if len(results) % 5 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

    # Final Save to CSV (in case the total number of questions is not a multiple of 5)
    final_df = pd.DataFrame(results)
    final_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSUCCESS! Unsloth inference completed. File generated: {OUTPUT_CSV}")

Starting optimized Unsloth inference on 643 questions...


100%|██████████| 643/643 [3:48:35<00:00, 21.33s/it]


SUCCESS! Unsloth inference completed. File generated: ./Team1/results/model2_finetuned.csv
